# VoxSentinel AI — Hindi / Hinglish ASR benchmark
Prepared configuration for VoxSentinel AI. **No model is trained by the project by default.**
Run top-to-bottom in Colab (T4/A100). Every section prints what it actually did.


## 1. Install

In [ ]:
!pip -q install "faster-whisper==1.0.*" "datasets==2.20.*" "jiwer==3.*" soundfile
import faster_whisper, jiwer; print("ok", faster_whisper.__version__)

## 2. Data (eval only) — Common Voice 17 hi, FLEURS hi, internal Hinglish clips

In [ ]:
from datasets import load_dataset
cv = load_dataset("mozilla-foundation/common_voice_17_0", "hi", split="test", token=True)
print(len(cv), cv.features)
fleurs = load_dataset("google/fleurs", "hi_in", split="test", token=True)
print(len(fleurs))

## 3. Run the models under consideration

In [ ]:
from faster_whisper import WhisperModel
for size in ["tiny", "small", "medium"]:
    m = WhisperModel(size, device="cuda", compute_type="float16")
    segs, info = m.transcribe("sample_hi.wav", language="hi")
    print(size, "->", " ".join(s.text for s in segs)[:120], "| p=", round(info.language_probability, 2))

## 4. Score WER/CER with normalisation

In [ ]:
import jiwer
norm = jiwer.Compose([jiwer.ToLowerCase(), jiwer.RemovePunctuation(),
                      jiwer.Strip(), jiwer.ReduceToListOfListOfWords()])
def score(ref, hyp):
    return {"wer": jiwer.wer(ref, hyp, truth_transform=norm, hypothesis_transform=norm),
            "cer": jiwer.cer(ref, hyp, truth_transform=norm, hypothesis_transform=norm)}
print(score("तुम्हारे बेटे को हमने किडनैप कर लिया है", "तुम्हारे बेटे को हमने किडनैप कर लिया है"))

## 5. Hinglish reality check
Report code-mixed clips separately. In this project `tiny` mangled a Hinglish ransom line into nonsense, which then made the classifier call it legitimate — ASR error is a first-class failure mode, not a footnote.